In [0]:
# BLOQUE 1 — Parámetros del snapshot GTFS a cargar en Bronze

import re

catalog = "workspace"
schema = "gtfs_bogota"
staging_volume_path = f"/Volumes/{catalog}/{schema}/staging_gtfs"

# La procedencia llega desde Raw como versión del proveedor y hash inmutable.
dbutils.widgets.text("feed_version", "2025-10-28", "Versión GTFS")
dbutils.widgets.text(
    "snapshot_sha256",
    "07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952",
    "SHA-256 del snapshot",
)
feed_version = dbutils.widgets.get("feed_version").strip()
snapshot_sha256 = dbutils.widgets.get("snapshot_sha256").strip().lower()

if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", feed_version):
    raise ValueError("feed_version debe tener el formato AAAA-MM-DD.")
if not re.fullmatch(r"[0-9a-f]{64}", snapshot_sha256):
    raise ValueError("snapshot_sha256 debe ser un hash SHA-256 de 64 caracteres.")

staging_snapshot_path = (
    f"{staging_volume_path}/gtfs_static/"
    f"feed_version={feed_version}/"
    f"sha256={snapshot_sha256}"
)

print(f"Snapshot de Staging: {staging_snapshot_path}")
display(dbutils.fs.ls(staging_snapshot_path))

In [0]:
# BLOQUE 2 — Contrato mínimo de las entidades GTFS de referencia

# El contrato exige solo columnas obligatorias de GTFS.
# Las columnas opcionales también se preservan como texto en Bronze.
reference_entities = [
    {
        "file_name": "agency.txt",
        "table_name": "bronze_gtfs_agency",
        "is_required": True,
        "required_columns": ["agency_name", "agency_url", "agency_timezone"],
    },
    {
        "file_name": "stops.txt",
        "table_name": "bronze_gtfs_stops",
        "is_required": True,
        "required_columns": ["stop_id", "stop_name", "stop_lat", "stop_lon"],
    },
    {
        "file_name": "trips.txt",
        "table_name": "bronze_gtfs_trips",
        "is_required": True,
        "required_columns": ["route_id", "service_id", "trip_id"],
    },
    {
        "file_name": "calendar.txt",
        "table_name": "bronze_gtfs_calendar",
        "is_required": True,
        "required_columns": ["service_id", "start_date", "end_date"],
    },
    {
        "file_name": "calendar_dates.txt",
        "table_name": "bronze_gtfs_calendar_dates",
        "is_required": True,
        "required_columns": ["service_id", "date", "exception_type"],
    },
    {
        "file_name": "frequencies.txt",
        "table_name": "bronze_gtfs_frequencies",
        "is_required": False,
        "required_columns": ["trip_id", "start_time", "end_time", "headway_secs"],
    },
    {
        "file_name": "fare_attributes.txt",
        "table_name": "bronze_gtfs_fare_attributes",
        "is_required": False,
        "required_columns": ["fare_id", "price", "currency_type", "payment_method", "transfers"],
    },
    {
        "file_name": "fare_rules.txt",
        "table_name": "bronze_gtfs_fare_rules",
        "is_required": False,
        "required_columns": ["fare_id"],
    },
    {
        "file_name": "feed_info.txt",
        "table_name": "bronze_gtfs_feed_info",
        "is_required": False,
        "required_columns": [
            "feed_publisher_name",
            "feed_publisher_url",
            "feed_lang",
        ],
    },
]

for entity in reference_entities:
    print(f"{entity['file_name']} -> {catalog}.{schema}.{entity['table_name']}")

In [0]:
# BLOQUE 3 — Funciones reutilizables de validación y escritura Bronze

import csv
from pathlib import PurePosixPath

from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    LongType,
    StringType,
    StructField,
    StructType,
)

def read_gtfs_header(source_path: str) -> list[str]:
    """Lee el encabezado UTF-8 del archivo y preserva sus nombres originales."""
    with open(source_path, "r", encoding="utf-8-sig", newline="") as source_file:
        header = next(csv.reader(source_file), None)

    if not header or any(not column.strip() for column in header):
        raise RuntimeError(f"Encabezado inválido o vacío en {source_path}.")
    if len(header) != len(set(header)):
        raise RuntimeError(f"El archivo tiene columnas duplicadas: {source_path}.")

    return header


def load_entity_to_bronze(entity: dict) -> tuple:
    """Valida un archivo GTFS y lo agrega una sola vez para cada hash fuente."""
    source_path = f"{staging_snapshot_path}/{entity['file_name']}"
    table_name = f"{catalog}.{schema}.{entity['table_name']}"
    source_columns = read_gtfs_header(source_path)

    missing_columns = sorted(set(entity['required_columns']) - set(source_columns))
    if missing_columns:
        raise RuntimeError(
            f"Contrato GTFS incumplido en {entity['file_name']}. "
            f"Faltan columnas: {', '.join(missing_columns)}"
        )

    # Bronze conserva todos los campos fuente como texto; Silver hará los casteos.
    source_schema = StructType([
        StructField(column, StringType(), True)
        for column in source_columns
    ])

    source_df = (
        spark.read
        .schema(source_schema)
        .option("header", True)
        .option("mode", "FAILFAST")
        .csv(source_path)
    )

    bronze_df = (
        source_df
        .withColumn("source_feed_version", F.lit(feed_version))
        .withColumn("source_sha256", F.lit(snapshot_sha256))
        .withColumn("source_file_path", F.lit(source_path))
        .withColumn("bronze_loaded_at_utc", F.current_timestamp())
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    try:
        source_row_count = bronze_df.count()

        if not spark.catalog.tableExists(table_name):
            (
                bronze_df.limit(0)
                .write
                .format("delta")
                .saveAsTable(table_name)
            )

        snapshot_already_loaded = (
            spark.table(table_name)
            .filter(F.col("source_sha256") == snapshot_sha256)
            .limit(1)
            .count() > 0
        )

        if not snapshot_already_loaded:
            # Bronze conserva la evolución publicada por el proveedor.
            # mergeSchema se limita a esta escritura; no activamos autoMerge global.
            (
                bronze_df
                .write
                .format("delta")
                .option("mergeSchema", "true")
                .mode("append")
                .saveAsTable(table_name)
            )

        return (
            entity['file_name'],
            table_name,
            source_row_count,
            snapshot_already_loaded,
        )
    finally:
        bronze_df.unpersist()


load_result_schema = StructType([
    StructField("source_file_name", StringType(), False),
    StructField("bronze_table_name", StringType(), False),
    StructField("source_row_count", LongType(), False),
    StructField("snapshot_already_loaded", BooleanType(), False),
])

In [0]:
# BLOQUE 4 — Carga idempotente de entidades de referencia en Delta Bronze

# GTFS permite archivos opcionales. Registramos su ausencia sin detener
# el pipeline ni inventar filas para una versión que no los publicó.
available_staging_files = {
    file.name
    for file in dbutils.fs.ls(staging_snapshot_path)
}

missing_required_files = [
    entity["file_name"]
    for entity in reference_entities
    if entity["is_required"]
    and entity["file_name"] not in available_staging_files
]
if missing_required_files:
    raise RuntimeError(
        "Faltan archivos GTFS requeridos: "
        + ", ".join(missing_required_files)
    )

entities_to_load = [
    entity
    for entity in reference_entities
    if entity["file_name"] in available_staging_files
]
skipped_optional_files = [
    entity["file_name"]
    for entity in reference_entities
    if not entity["is_required"]
    and entity["file_name"] not in available_staging_files
]

load_results = [
    load_entity_to_bronze(entity)
    for entity in entities_to_load
]

load_results_df = spark.createDataFrame(
    load_results,
    schema=load_result_schema,
)

print("Archivos opcionales no publicados: " + str(skipped_optional_files))
display(load_results_df.orderBy("source_file_name"))

In [0]:
# BLOQUE 5 — Verificación inicial de paradas cargadas

bronze_stops_table = f"{catalog}.{schema}.bronze_gtfs_stops"

display(
    spark.table(bronze_stops_table)
    .filter(F.col("source_sha256") == snapshot_sha256)
    .orderBy("stop_id")
    .limit(20)
)

display(spark.sql(f"describe detail {bronze_stops_table}"))